# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset metadata is described via a Croissant schema:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

*Note: All references to data entities are made via their `@id` fields as per best practices.*

In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata (schema)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Loaded dataset:")
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")
print(f"Keywords: {', '.join(metadata.keywords) if hasattr(metadata, 'keywords') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references with the Croissant schema. This will help you identify how the tables and variables are organized in the dataset.

In [ ]:
# List all available record sets and their IDs
print('Available RecordSets:')
recordsets = metadata.recordSets if hasattr(metadata, 'recordSets') else getattr(metadata, 'recordSet', [])
if not recordsets:
    # Try to fall back to dataset.record_sets (mlcroissant 0.1.18+)
    recordsets = dataset.record_sets if hasattr(dataset, 'record_sets') else []
for i, rs in enumerate(recordsets):
    print(f"  {i+1}. Name: {rs.name if hasattr(rs, 'name') else 'N/A'} | @id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', 'N/A')}")

# Example: For this dataset, we examine the actual schema to find the primary record set, which is usually the table with experimental data.
# We'll attempt to print more information about the first record set.
if recordsets:
    first_rs = recordsets[0]
    rs_id = first_rs['@id'] if isinstance(first_rs, dict) and '@id' in first_rs else getattr(first_rs, '@id', None)
    print(f"\nFields for RecordSet '@id': {rs_id}")
    fields = first_rs['field'] if isinstance(first_rs, dict) and 'field' in first_rs else getattr(first_rs, 'fields', [])
    for f in fields:
        field_id = f['@id'] if isinstance(f, dict) else getattr(f, '@id', None)
        fname = f['name'] if isinstance(f, dict) and 'name' in f else getattr(f, 'name', field_id)
        dtype = f['dataType'] if isinstance(f, dict) and 'dataType' in f else getattr(f, 'dataType', 'unknown')
        print(f"  - {fname} (@id: {field_id}, dataType: {dtype})")

In [ ]:
# If you want to visually inspect a few records for a RecordSet, specify the '@id' string.
# We'll try to automatically find the first record set's '@id' as an example.

record_set_id = None
# Find the primary record set '@id':
for rs in recordsets:
    rid = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
    if rid:
        record_set_id = rid
        break

print(f"\nSample records from record set: {record_set_id}")
for idx, record in enumerate(dataset.records(record_set=record_set_id)):
    print(f"Record {idx}:", record)
    if idx >= 2:
        break

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. Use record set and field `@id` references determined above.

In [ ]:
# Build a list of record set @ids to extract (for this dataset, typically only one main table)
record_sets_ids = []
for rs in recordsets:
    rid = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None)
    if rid:
        record_sets_ids.append(rid)
# For demonstration, proceed with the first
dataframes = {}
for record_set_id in record_sets_ids:
    print(f"\nLoading records for RecordSet '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"No records found for {record_set_id}.")
        continue
    df = pd.DataFrame(records)
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head())
    dataframes[record_set_id] = df

# For ease of further processing, set main_record_set_id for use.
main_record_set_id = record_sets_ids[0] if record_sets_ids else None

## 4. Exploratory Data Analysis (EDA)
Now, let's apply some data processing to the tabular data extracted. We'll pick a numeric field (by `@id`). You may want to update `numeric_field_id` and `group_field_id` to match the actual field names or IDs in your dataset.

*Common steps: filter out low values, normalize a column, group by a categorical variable, etc.*

> **Tip:** Use column names printed above to decide which field to analyze.

In [ ]:
# Example: Replace with actual field '@id' strings or DataFrame column names seen above

df = dataframes[main_record_set_id]
# Example field names, adjust as appropriate to your dataset
# For demonstration, we'll detect a likely numeric column
numeric_col_candidates = [col for col in df.columns if df[col].dtype in ('int64', 'float64')] # or add type inference
if not numeric_col_candidates:
    # Try to coerce columns to numeric and pick one
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
            numeric_col_candidates.append(col)
        except:
            pass
# For this demo we use the first numeric detected field (often "MW_kDa" or similar)
numeric_field_id = numeric_col_candidates[0] if numeric_col_candidates else df.columns[0]  # fallback
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Filtering: keep records with value > threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean): {len(filtered_df)} records.")

# Normalization (Z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by a categorical field (if available)
categorical_candidates = [col for col in df.columns if df[col].nunique() < 10 and col != numeric_field_id]
group_field = categorical_candidates[0] if categorical_candidates else None

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical field for grouping found.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and (if a suitable categorical field is available) create grouped boxplots.

*Note: For more advanced visualizations, consider using seaborn or plotly.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution histogram
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If categorical field available, make a boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, you:

- Loaded and reviewed the FAIR^2 dataset metadata using `mlcroissant`.
- Explored and extracted record sets via their Croissant `@id` fields.
- Loaded records into DataFrames and performed initial exploratory data analysis (EDA): filtering, normalization, and grouping.
- Created basic visualizations of quantitative fields.

This workflow demonstrates best practices for FAIR data access and structured analysis. For customized exploration, drill into specific field `@id`s or add your domain-specific analysis as needed!